# Escalation-Stage Model — Ordinal Stage Filter + Clip-Referenced Escalation Pathway

Trains and evaluates the escalation-stage architecture decided on
2026-11-09 / built 2026-11-16 (`docs/roadmap.md`). Input is the
per-window DINOv2 + OpenCLIP sequences written by
`sliding_window_feature_extraction.ipynb`; output is a causal
Normal / Escalating / Critical posterior for every 1-second step of a
clip (the green / amber / red indicator).

**The two designed mechanisms (the novelty claim rests on these, not on
the frozen backbones):**

1. **Ordinal stage filter.** Two evidence signals per window -- Critical
   evidence and Escalation evidence -- are combined into rank-consistent
   ordinal emissions and passed through the forward (causal) filter of a
   3-state HMM whose transition matrix is *banded*: a clip cannot go
   Normal -> Critical without passing through Escalating. Trained from
   video-level labels only. "Escalating" gets a precise meaning through
   *self-anticipation*: the Escalation evidence is trained to predict
   that the model's own Critical stage will fire within `horizon_s`
   seconds (pseudo-onsets, no frame labels used).
2. **Clip-referenced escalation pathway.** Escalation evidence is
   computed only from how the current window differs from the running
   mean of everything seen earlier *in the same clip*. Anything constant
   across a clip (watermark, letterboxing, channel logo, compression
   signature) cancels by construction, so the amber light cannot be
   switched on by what a video looks like -- only by how it is changing.
   Motivated by the Pre-Event Separability confound (`docs/roadmap.md`).

**Arms trained (same data, same seeds) so every claim has a control:**

| arm | what it is | what it tests |
|---|---|---|
| `baseline` | one causal GRU + sigmoid, top-k MIL | is there usable signal at all |
| `ordinal_abs` | mechanism 1, escalation evidence from absolute features | does the ordinal filter + self-anticipation help |
| `full` | mechanism 1 + mechanism 2 | does clip-referencing remove static-source leakage |
| `full_adv` | `full` + movie-vs-YouTube adversary on the Critical pathway | does removing the medium shortcut help (riot clips are 95% YouTube, normal clips ~52%) |

**Read before quoting any number from this notebook:**
- Only **25 of the 101** annotated riot test clips have more than 5 s of
  calm footage before the riot starts (59 start mid-riot at frame 0).
  Every lead-time number here is computed on that small set -- report
  `n` next to it, every time.
- Mechanism 2 protects the *Escalation* evidence. The Critical pathway
  still sees absolute content, like every published MIL detector, and
  can still exploit static source cues. The "far pre-onset" check below
  measures exactly that leakage rather than assuming it away.
- Riot-only vs. Normal, as before: not the 6-class XD-Violence benchmark.

**Requires:** `.npz` window features on Drive. Works on a partial
extraction (it uses whatever clips exist and prints the counts), but
needs a reasonable number of *test-split normal* clips for the
false-alarm side of the evaluation.

## Step 0 — Setup

In [ ]:
!pip install scikit-learn requests matplotlib -q

In [ ]:
import json
import math
import os
import pickle
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/CVPHG'
FEAT_DIR = f'{ROOT}/sliding_window_features'
MANIFESTS = [
    f'{ROOT}/xdviolence_riot/manifest_xdviolence_riot.csv',
    f'{ROOT}/xdviolence_normal/manifest_xdviolence_normal.csv',
]
OUT_DIR = f'{ROOT}/escalation_runs'
ANNOTATIONS_SOURCE = 'https://roc-ng.github.io/XD-Violence/images/annotations.txt'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
CFG = dict(
    # --- what to run ---
    arms=['baseline', 'ordinal_abs', 'full', 'full_adv'],
    seeds=[0, 1, 2],            # project rule: no single-seed claims
    # --- training ---
    epochs=40,
    warmup_epochs=8,            # MIL only; self-anticipation + adversary switch on after this
    ramp_epochs=5,
    batch_size=16,
    lr=5e-4,
    weight_decay=1e-2,
    max_len=600,                # training crops, in windows (~10 min)
    val_frac=0.15,
    # --- model ---
    enc_dim=256,
    gru_dim=128,
    dropout=0.3,
    topk_frac=1 / 16,
    horizon_s=10.0,             # Escalating == "Critical expected within this many seconds"
    antic_weight=1.0,
    adv_weight=0.3,
    rel_scale=2.0,
    # --- evaluation ---
    min_pre_s=5.0,              # an onset needs this much calm footage before it to count
    warn_window_s=30.0,         # an alarm raised earlier than this before onset is "premature"
    near_s=10.0,
    far_s=20.0,
    video_fa_target=0.10,       # calibrated operating point: <= this share of validation normal videos ever alarm
    tau_floor=0.05,             # never calibrate an alarm threshold below this posterior
    c_miss=1.0,
    c_fa=0.2,                   # a missed/late riot costs 5x a premature alarm
    max_delay_s=10.0,
)

## Step 1 — Load window features

Each clip is a sequence of windows. DINOv2 and OpenCLIP embeddings are
L2-normalised separately and concatenated (1536-d). Windows whose frame
could not be read during extraction (all-zero rows) are dropped. A
single cache file is kept on Drive so later runs do not re-read
thousands of small `.npz` files.

In [ ]:
def _l2(a):
    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-8)


def load_clip(npz_path):
    d = np.load(npz_path)
    ws = d['window_start'].astype(np.int64)
    we = d['window_end'].astype(np.int64)
    ok = we > ws
    if ok.sum() == 0:
        return None
    feats = np.concatenate(
        [_l2(d['dinov2'][ok].astype(np.float32)), _l2(d['openclip'][ok].astype(np.float32))], axis=1)
    return dict(
        feats=feats.astype(np.float16),
        centers=(ws[ok] + we[ok]) // 2,   # the frame that was actually embedded
        fps=float(d['fps']),
        total_frames=int(d['total_frames']),
    )


manifest = pd.concat([pd.read_csv(p) for p in MANIFESTS], ignore_index=True)
manifest = manifest[['id', 'label', 'split']].drop_duplicates('id').reset_index(drop=True)

cache_path = os.path.join(OUT_DIR, 'window_cache.pkl')
cache = {}
if os.path.exists(cache_path):
    with open(cache_path, 'rb') as fh:
        cache = pickle.load(fh)

on_disk = {f[:-4] for f in os.listdir(FEAT_DIR) if f.endswith('.npz')}
todo = [v for v in manifest['id'] if v in on_disk and v not in cache]
print(f"{len(on_disk)} .npz files on disk, {len(cache)} already cached, {len(todo)} to read")
for i, vid in enumerate(todo, 1):
    try:
        rec = load_clip(os.path.join(FEAT_DIR, vid + '.npz'))
    except Exception as e:  # e.g. a file still being written by a running extraction
        print(f"  skipped {vid}: {e}")
        continue
    if rec is not None:
        cache[vid] = rec
    if i % 200 == 0:
        print(f"  {i}/{len(todo)} read")
if todo:
    with open(cache_path, 'wb') as fh:
        pickle.dump(cache, fh, protocol=4)

clips = []
for row in manifest.itertuples():
    if row.id in cache:
        clips.append(dict(
            id=row.id, y=int(row.label == 'riot'), split=row.split,
            medium=int(row.id.startswith('v=')),        # 1 = YouTube, 0 = movie
            source=row.id.split('__#')[0],              # video of origin
            **cache[row.id]))

for c in clips:
    c['times'] = c['centers'] / c['fps']               # seconds; when each window is observed

IN_DIM = clips[0]['feats'].shape[1]
STRIDE_S = float(np.median([np.median(np.diff(c['times'])) for c in clips if len(c['times']) > 2]))
H_WIN = max(1, int(round(CFG['horizon_s'] / STRIDE_S)))

summary = pd.DataFrame([dict(label='riot' if c['y'] else 'normal', split=c['split'],
                             medium='youtube' if c['medium'] else 'movie',
                             windows=len(c['times'])) for c in clips])
print(f"\n{len(clips)} clips loaded, {summary['windows'].sum()} windows, "
      f"stride {STRIDE_S:.2f}s, horizon {H_WIN} windows")
print(summary.groupby(['label', 'split', 'medium']).agg(clips=('windows', 'size'), windows=('windows', 'sum')))

n_test_normal = sum(1 for c in clips if c['split'] == 'test' and c['y'] == 0)
n_train_normal = sum(1 for c in clips if c['split'] == 'train' and c['y'] == 0)
assert n_test_normal > 0 and n_train_normal > 0, (
    "No normal clips extracted for one of the splits yet -- the false-alarm side of the "
    "evaluation cannot run. Let sliding_window_feature_extraction.ipynb get further first.")
if n_test_normal < 100 or n_train_normal < 300:
    print(f"\nWARNING: only {n_train_normal} train / {n_test_normal} test normal clips extracted so far. "
          "Results below are provisional -- false-alarm numbers especially. Label them as partial-data.")

In [ ]:
# Official split is kept. A validation set is carved out of official-train
# BY SOURCE VIDEO (parts of one YouTube video / movie never straddle
# train and val). It is used for early stopping and for calibrating alarm
# thresholds -- never the test split.
split_rng = random.Random(0)
train_clips, val_clips = [], []
for y in (0, 1):
    pool = [c for c in clips if c['split'] == 'train' and c['y'] == y]
    sources = sorted({c['source'] for c in pool})
    split_rng.shuffle(sources)
    val_sources = set(sources[:max(1, int(round(len(sources) * CFG['val_frac'])))])
    for c in pool:
        (val_clips if c['source'] in val_sources else train_clips).append(c)
test_clips = [c for c in clips if c['split'] == 'test']


def _count(cs):
    return f"{sum(c['y'] for c in cs)} riot / {sum(1 - c['y'] for c in cs)} normal"


print(f"train: {_count(train_clips)}\nval:   {_count(val_clips)}\ntest:  {_count(test_clips)}")

## Step 2 — Frame-level ground truth (official `annotations.txt`)

Only used for **evaluation** on the official test split. Training never
sees it. One annotation entry carries a stray `.mp4` in its name -- that
was the single unmatched clip in the 2026-10-19 apples-to-apples run;
it is normalised here.

In [ ]:
def load_annotations(source):
    if source.startswith('http'):
        import requests
        resp = requests.get(source, timeout=60)
        resp.raise_for_status()
        text = resp.text
    else:
        with open(source) as fh:
            text = fh.read()
    out = {}
    for line in text.splitlines():
        parts = line.split()
        if not parts:
            continue
        name = parts[0][:-4] if parts[0].endswith('.mp4') else parts[0]
        nums = [int(x) for x in parts[1:]]
        out[name] = [(nums[i], nums[i + 1]) for i in range(0, len(nums) - 1, 2)]
    return out


annotations = load_annotations(ANNOTATIONS_SOURCE)
unmatched = []
for c in test_clips:
    c['intervals'] = None
    if c['y'] == 1:
        c['intervals'] = annotations.get(os.path.splitext(c['id'])[0])
        if c['intervals'] is None:
            unmatched.append(c['id'])
riot_test = [c for c in test_clips if c['y'] == 1 and c['intervals']]
print(f"{len(riot_test)} riot test clips matched to annotations; {len(unmatched)} unmatched (excluded): {unmatched[:5]}")


def frame_truth(c):
    y = np.zeros(max(c['total_frames'], int(c['centers'][-1]) + 1), dtype=np.int8)
    for s, e in (c['intervals'] or []):
        y[max(0, s):min(len(y), e + 1)] = 1
    return y


def window_scores_to_frames(c, scores):
    """Each frame takes the score of the window whose embedded frame is nearest."""
    bounds = (c['centers'][:-1] + c['centers'][1:]) / 2.0
    n = max(c['total_frames'], int(c['centers'][-1]) + 1)
    return scores[np.searchsorted(bounds, np.arange(n))]


def onset_events(c):
    """(calm_span_start_s, onset_s, interval_end_s, is_first) per annotated interval."""
    ev, prev_end = [], 0.0
    for k, (s, e) in enumerate(sorted(c['intervals'])):
        ev.append((prev_end, s / c['fps'], e / c['fps'], k == 0))
        prev_end = e / c['fps']
    return ev


first_onsets = [(c, ev) for c in riot_test for ev in onset_events(c)
                if ev[3] and ev[1] - ev[0] >= CFG['min_pre_s']]
all_onsets = [(c, ev) for c in riot_test for ev in onset_events(c)
              if ev[1] - ev[0] >= CFG['min_pre_s']]
onset0 = sum(1 for c in riot_test if min(s for s, _ in c['intervals']) == 0)
print(f"{onset0}/{len(riot_test)} riot test clips are riot from frame 0.")
print(f"Lead-time evaluation set: {len(first_onsets)} first onsets with >= {CFG['min_pre_s']:.0f}s of calm before them "
      f"({len(all_onsets)} if re-onsets after a calm gap are included).")

## Step 3 — How big is the confound in the raw features? (no model yet)

A plain per-window logistic probe is trained on official-train windows
with nothing but the video-level label, i.e. the naive thing. Then, on
test:
- **near**: windows in the last `near_s` seconds before a riot starts
- **far**: windows more than `far_s` seconds before it starts
- each compared against windows of normal test videos (and against
  YouTube-only normals, since riot clips are almost all YouTube).

If "far" windows -- calm footage long before anything happens -- are
already separable from normal videos, that is static source/scene
leakage, not escalation. The gap `near - far` is the part that is
specific to the run-up. The same two numbers are reported for every
trained arm in Step 6.

In [ ]:
def pre_onset_windows(score_of):
    """score_of: clip -> per-window score array. Scores of the calm windows
    just before a riot starts (near) and long before it (far)."""
    near, far, t_cap = [], [], 0.0
    for c, (span_start, onset, _end, _first) in first_onsets:
        s, t = score_of(c), c['times']
        near.append(s[(t >= max(span_start, onset - CFG['near_s'])) & (t < onset)])
        far.append(s[(t >= span_start) & (t < onset - CFG['far_s'])])
        t_cap = max(t_cap, onset)
    near = np.concatenate(near) if near else np.array([])
    far = np.concatenate(far) if far else np.array([])
    return near, far, t_cap


def pre_onset_sets(score_of):
    """near / far AUCs against windows of normal test videos."""
    near, far, t_cap = pre_onset_windows(score_of)
    res = dict(n_near=len(near), n_far=len(far))
    for tag, keep in (('all', lambda c: True), ('yt', lambda c: c['medium'] == 1)):
        ctrl = [score_of(c)[c['times'] <= t_cap] for c in test_clips if c['y'] == 0 and keep(c)]
        ctrl = np.concatenate(ctrl) if ctrl else np.array([])
        for name, pos in (('near', near), ('far', far)):
            ok = len(pos) >= 5 and len(ctrl) >= 5
            res[f'auc_{name}_{tag}'] = float(roc_auc_score(
                np.r_[np.ones(len(pos)), np.zeros(len(ctrl))], np.r_[pos, ctrl])) if ok else float('nan')
    return res


def sample_windows(cs, per_clip, rng):
    X, y, m = [], [], []
    for c in cs:
        idx = rng.choice(len(c['times']), size=min(per_clip, len(c['times'])), replace=False)
        X.append(c['feats'][idx].astype(np.float32))
        y.append(np.full(len(idx), c['y']))
        m.append(np.full(len(idx), c['medium']))
    return np.concatenate(X), np.concatenate(y), np.concatenate(m)


probe_rng = np.random.default_rng(0)
Xp, yp, mp = sample_windows(train_clips + val_clips, 20, probe_rng)
probe = LogisticRegression(C=1.0, max_iter=300).fit(Xp, yp)
raw_probe = pre_onset_sets(lambda c: probe.predict_proba(c['feats'].astype(np.float32))[:, 1])
print("Raw-feature probe (video-label supervision only):")
print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in raw_probe.items()}, indent=1))

# Medium shortcut: can a single normal window tell you it came from YouTube rather than a movie?
norm_tr = [c for c in train_clips + val_clips if c['y'] == 0]
norm_te = [c for c in test_clips if c['y'] == 0]
MEDIUM_OK = min(sum(c['medium'] for c in norm_tr), sum(1 - c['medium'] for c in norm_tr)) >= 20
if MEDIUM_OK and len({c['medium'] for c in norm_te}) == 2:
    Xm, _, mm = sample_windows(norm_tr, 20, probe_rng)
    Xt, _, mt = sample_windows(norm_te, 20, probe_rng)
    med_probe = LogisticRegression(C=1.0, max_iter=300).fit(Xm, mm)
    print(f"Movie-vs-YouTube from one normal window: AUC {roc_auc_score(mt, med_probe.predict_proba(Xt)[:, 1]):.3f} "
          "(1.0 = medium is trivially readable from the features)")
else:
    print("Not enough movie AND YouTube normal clips yet for the medium probe / adversary arm.")

## Step 4 — The model

In [ ]:
class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None


def hmm_forward_filter(emis, A, pi):
    """Causal filtered posterior over stages. emis: (B, T, K) emission
    probabilities, A: (K, K) row-stochastic transitions, pi: (K,) initial."""
    alpha = pi.unsqueeze(0) * emis[:, 0]
    alpha = alpha / alpha.sum(-1, keepdim=True)
    outs = [alpha]
    for t in range(1, emis.size(1)):
        alpha = emis[:, t] * torch.matmul(alpha, A)
        alpha = alpha / alpha.sum(-1, keepdim=True)
        outs.append(alpha)
    return torch.stack(outs, dim=1)


def clip_referenced(x):
    """x_t minus the mean of every earlier window of the same clip (causal).
    The first window has no reference and is zeroed. Anything constant
    across the clip cancels exactly."""
    T = x.size(1)
    prev_sum = torch.cumsum(x, dim=1) - x
    cnt = torch.arange(T, device=x.device, dtype=x.dtype).clamp(min=1.0).view(1, T, 1)
    has_ref = (torch.arange(T, device=x.device) > 0).to(x.dtype).view(1, T, 1)
    return (x - prev_sum / cnt) * has_ref


class EscalationNet(nn.Module):
    BAND = [[1, 1, 0], [1, 1, 1], [0, 1, 1]]   # stage k may only move to k-1, k, k+1

    def __init__(self, in_dim, arm, cfg):
        super().__init__()
        self.arm = arm
        self.ordinal = arm != 'baseline'
        self.clip_ref = arm in ('full', 'full_adv')
        self.adversary = arm == 'full_adv'
        self.rel_scale = cfg['rel_scale']
        D, H, p = cfg['enc_dim'], cfg['gru_dim'], cfg['dropout']

        # Critical pathway: absolute content, causal.
        self.enc_crit = nn.Sequential(nn.Linear(in_dim, D), nn.LayerNorm(D), nn.GELU(), nn.Dropout(p))
        self.gru_crit = nn.GRU(D, H, batch_first=True)
        self.head_crit = nn.Linear(H, 1)

        if self.ordinal:
            # Escalation pathway. No input normalisation on purpose: when it is
            # fed clip-referenced features, the *size* of the change is signal.
            self.enc_esc = nn.Sequential(nn.Linear(in_dim, D), nn.GELU(), nn.Dropout(p), nn.Linear(D, D), nn.GELU())
            self.gru_esc = nn.GRU(D, H, batch_first=True)
            self.head_esc = nn.Linear(H, 1)
            self.register_buffer('band', torch.tensor(self.BAND, dtype=torch.bool))
            stay = torch.tensor([[0.90, 0.10, 0.0], [0.05, 0.90, 0.05], [0.0, 0.10, 0.90]])
            self.trans_logits = nn.Parameter(torch.log(stay.clamp(min=1e-8)))
            self.init_logits = nn.Parameter(torch.zeros(3))
        if self.adversary:
            self.disc = nn.Sequential(nn.Linear(D, 64), nn.GELU(), nn.Linear(64, 1))

    def transition_matrix(self):
        return torch.softmax(self.trans_logits.masked_fill(~self.band, float('-inf')), dim=-1)

    def forward(self, x, grl_lambda=0.0):
        e = self.enc_crit(x)
        h, _ = self.gru_crit(e)
        rho = torch.sigmoid(self.head_crit(h).squeeze(-1))          # Critical evidence
        out = {}
        if not self.ordinal:
            out.update(red=rho, alarm=rho, esc=None, stages=None)
        else:
            inp = clip_referenced(x) * self.rel_scale if self.clip_ref else x
            he, _ = self.gru_esc(self.enc_esc(inp))
            eps = torch.sigmoid(self.head_esc(he).squeeze(-1))      # Escalation evidence
            # Rank-consistent ordinal emissions: P(>=Critical) <= P(>=Escalating) always.
            emis = torch.stack([(1 - rho) * (1 - eps), (1 - rho) * eps, rho], dim=-1)
            emis = emis * (1 - 1e-3) + 1e-3 / 3
            # Run on CPU: thousands of tiny 3-state updates cost less there than as GPU kernel launches.
            alpha = hmm_forward_filter(emis.cpu(), self.transition_matrix().cpu(),
                                       torch.softmax(self.init_logits, dim=0).cpu()).to(emis.device)
            out.update(red=alpha[..., 2], alarm=(1 - alpha[..., 0]).clamp(0, 1), esc=eps, stages=alpha)
        if self.adversary:
            out['src_logit'] = self.disc(_GradReverse.apply(e, grl_lambda)).squeeze(-1)
        return out


def topk_mean(v, mask, frac):
    k = torch.clamp((mask.sum(1).float() * frac).ceil().long(), min=1)
    vs, _ = torch.sort(v.masked_fill(~mask, -1.0), dim=1, descending=True)
    sel = torch.arange(v.size(1), device=v.device)[None, :] < k[:, None]
    return (vs * sel).sum(1) / k.float()


def lookahead_max(v, H):
    """out[t] = max(v[t : t + H + 1])"""
    return F.max_pool1d(F.pad(v, (0, H), value=0.0).unsqueeze(1), kernel_size=H + 1, stride=1).squeeze(1)


def compute_losses(out, y, mask, medium, cfg, sched):
    eps = 1e-6
    red, alarm = out['red'].clamp(eps, 1 - eps), out['alarm'].clamp(eps, 1 - eps)
    pos, neg = y == 1, y == 0
    fm = mask.float()
    zero = red.new_zeros(())
    L = {}

    # Video-level MIL. Riot video: its top windows must reach Critical.
    # Normal video: every window is known Normal -- nothing above green anywhere.
    l_pos = -torch.log(topk_mean(red, mask, cfg['topk_frac'])[pos]).mean() if pos.any() else zero
    quiet = -torch.log(1 - alarm)
    l_neg = zero
    if neg.any():
        l_neg = (-torch.log(1 - topk_mean(alarm, mask, cfg['topk_frac'])[neg]).mean()
                 + ((quiet * fm).sum(1) / fm.sum(1))[neg].mean())
    L['mil'] = l_pos + l_neg

    # Self-anticipation: Escalation evidence should be on exactly when the
    # model's own Critical stage is about to fire. Windows that are already
    # Critical are down-weighted (amber is moot there), which also keeps the
    # riot-from-frame-0 clips from injecting an unlearnable target.
    L['antic'] = zero
    if out['esc'] is not None and pos.any():
        red_d = red.detach() * fm
        target = lookahead_max(red_d, H_WIN)
        w = (1 - red_d) * fm * pos.float()[:, None]
        bce = F.binary_cross_entropy(out['esc'].clamp(eps, 1 - eps), target, reduction='none')
        L['antic'] = (bce * w).sum() / w.sum().clamp(min=1.0)

    # Medium adversary: on NORMAL videos only, a discriminator tries to read
    # movie-vs-YouTube off the Critical pathway's features; the gradient is
    # reversed into the encoder so that it cannot.
    L['adv'] = zero
    if 'src_logit' in out and neg.any():
        tgt = medium.float()[:, None].expand_as(out['src_logit'])
        bce = F.binary_cross_entropy_with_logits(out['src_logit'], tgt, reduction='none')
        parts = []
        for m in (0, 1):
            w = fm * (neg & (medium == m)).float()[:, None]
            if w.sum() > 0:
                parts.append((bce * w).sum() / w.sum())
        if parts:
            L['adv'] = torch.stack(parts).mean()

    L['total'] = L['mil'] + sched * cfg['antic_weight'] * L['antic'] + L['adv']
    return L

## Step 5 — Training

In [ ]:
def collate(batch, max_len=None, rng=None):
    feats = []
    for c in batch:
        f = c['feats']
        if max_len is not None and len(f) > max_len:
            s = rng.randrange(0, len(f) - max_len + 1)
            f = f[s:s + max_len]
        feats.append(f)
    T = max(len(f) for f in feats)
    x = np.zeros((len(batch), T, feats[0].shape[1]), dtype=np.float32)
    mask = np.zeros((len(batch), T), dtype=bool)
    for i, f in enumerate(feats):
        x[i, :len(f)] = f
        mask[i, :len(f)] = True
    return (torch.from_numpy(x).to(DEVICE), torch.from_numpy(mask).to(DEVICE),
            torch.tensor([c['y'] for c in batch], device=DEVICE),
            torch.tensor([c['medium'] for c in batch], device=DEVICE))


@torch.no_grad()
def predict(model, cs, batch_size=16):
    """Per-clip numpy arrays: alarm (>= Escalating), red (Critical), esc, stages."""
    model.eval()
    order = sorted(range(len(cs)), key=lambda i: len(cs[i]['times']))
    preds = [None] * len(cs)
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        x, mask, _, _ = collate([cs[i] for i in idx])
        out = model(x)
        for j, i in enumerate(idx):
            n = len(cs[i]['times'])
            preds[i] = dict(
                alarm=out['alarm'][j, :n].cpu().numpy(), red=out['red'][j, :n].cpu().numpy(),
                esc=None if out['esc'] is None else out['esc'][j, :n].cpu().numpy(),
                stages=None if out['stages'] is None else out['stages'][j, :n].cpu().numpy())
    return preds


@torch.no_grad()
def validation_loss(model, cs, cfg):
    model.eval()
    tot, n, bag, ys = 0.0, 0, [], []
    for b in range(0, len(cs), 16):
        batch = cs[b:b + 16]
        x, mask, y, medium = collate(batch)
        out = model(x)
        tot += float(compute_losses(out, y, mask, medium, cfg, 1.0)['mil']) * len(batch)
        n += len(batch)
        bag.extend(topk_mean(out['red'], mask, cfg['topk_frac']).cpu().tolist())
        ys.extend(y.cpu().tolist())
    auc = roc_auc_score(ys, bag) if len(set(ys)) == 2 else float('nan')
    return tot / max(n, 1), auc


def train_arm(arm, seed, cfg, verbose=True):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rng = random.Random(seed)
    model = EscalationNet(IN_DIM, arm, cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched_lr = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    pos = [c for c in train_clips if c['y'] == 1]
    neg = [c for c in train_clips if c['y'] == 0]
    half = cfg['batch_size'] // 2
    steps = math.ceil(max(len(pos), len(neg)) / half)
    min_epoch = cfg['warmup_epochs'] + cfg['ramp_epochs']
    best = (float('inf'), None, -1)
    t0 = time.time()
    for epoch in range(cfg['epochs']):
        model.train()
        sched = float(np.clip((epoch - cfg['warmup_epochs'] + 1) / cfg['ramp_epochs'], 0.0, 1.0))
        rng.shuffle(pos); rng.shuffle(neg)
        agg = {}
        for s in range(steps):
            batch = ([pos[(s * half + i) % len(pos)] for i in range(half)]
                     + [neg[(s * half + i) % len(neg)] for i in range(half)])
            x, mask, y, medium = collate(batch, cfg['max_len'], rng)
            out = model(x, grl_lambda=sched * cfg['adv_weight'])
            L = compute_losses(out, y, mask, medium, cfg, sched)
            opt.zero_grad()
            L['total'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            for k, v in L.items():
                agg[k] = agg.get(k, 0.0) + float(v) / steps
        sched_lr.step()
        v_loss, v_auc = validation_loss(model, val_clips, cfg)
        if epoch + 1 >= min_epoch and v_loss < best[0]:
            best = (v_loss, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, epoch + 1)
        if verbose and (epoch % 5 == 4 or epoch == 0):
            print(f"  [{arm} seed {seed}] epoch {epoch + 1:>2}  mil {agg['mil']:.3f}  antic {agg['antic']:.3f}  "
                  f"adv {agg['adv']:.3f}  | val mil {v_loss:.3f}  val bag-AUC {v_auc:.3f}  ({time.time() - t0:.0f}s)")
    if best[1] is not None:
        model.load_state_dict(best[1])
    if verbose:
        print(f"  [{arm} seed {seed}] kept epoch {best[2]} (val mil {best[0]:.3f})")
    return model

## Step 6 — Evaluation

All on the **official test split**. Two alarm operating points:
`@0.5` (alarm = posterior of Escalating-or-worse >= 0.5) and `@cal`
(threshold set so that at most `video_fa_target` of *validation* normal
videos ever alarm). Nothing is tuned on test.

- **Frame-level AP / AUC** against `annotations.txt` (same metric as the
  2026-10-19 result, now with scores that vary over time).
- **Lead time**, per onset: an alarm counts as an *early warning* only
  if it switches on during the calm span before the riot and within
  `warn_window_s` of it. An alarm that was already on when the calm span
  began, or that came on earlier than that, is *premature* -- an
  always-amber light gets no credit. *Late* = switched on after onset;
  *miss* = never during the riot.
- **Average cost** (asymmetric, adapted from Achenchabe et al.'s
  cost-based early-classification criterion): a missed riot costs
  `c_miss`, a late one a share of it growing with the delay, a premature
  or false alarm `c_fa`.
- **Near / far pre-onset AUC** -- the confound check from Step 3,
  applied to each model's own outputs.

In [ ]:
def event_outcome(times, on, span_start, onset, end, warn_window):
    pre = np.where((times >= span_start) & (times < onset))[0]
    false_runs = 0
    if len(pre):
        seg = on[pre].astype(int)
        false_runs = int(((np.diff(np.r_[0, seg]) == 1).sum()) - (1 if seg[-1] else 0))
        if seg[-1]:
            j = len(seg) - 1
            while j > 0 and seg[j - 1]:
                j -= 1
            lead = float(onset - times[pre[j]])
            if j == 0 or lead > warn_window:
                return 'premature', lead, false_runs
            return 'early', lead, false_runs
    post = np.where((times >= onset) & (times <= end) & on)[0]
    if len(post):
        return 'late', float(times[post[0]] - onset), false_runs
    return 'miss', float('nan'), false_runs


def lead_time_report(events, preds_by_id, tau, cfg):
    kinds, leads, delays, false_runs, costs = [], [], [], 0, []
    for c, (span_start, onset, end, _first) in events:
        kind, val, fr = event_outcome(c['times'], preds_by_id[c['id']]['alarm'] >= tau,
                                      span_start, onset, end, cfg['warn_window_s'])
        kinds.append(kind)
        false_runs += fr
        if kind == 'early':
            leads.append(val); costs.append(0.0)
        elif kind == 'late':
            delays.append(val); costs.append(cfg['c_miss'] * min(val / cfg['max_delay_s'], 1.0))
        elif kind == 'miss':
            costs.append(cfg['c_miss'])
        else:
            costs.append(cfg['c_fa'])
    n = max(len(kinds), 1)
    rep = {k: kinds.count(k) / n for k in ('early', 'premature', 'late', 'miss')}
    rep.update(n_events=len(kinds), median_lead_s=float(np.median(leads)) if leads else float('nan'),
               median_delay_s=float(np.median(delays)) if delays else float('nan'),
               false_runs_in_calm=false_runs)
    return rep, costs


def evaluate(model, cfg):
    val_preds = predict(model, val_clips)
    test_preds = predict(model, test_clips)
    by_id = {c['id']: p for c, p in zip(test_clips, test_preds)}
    m = {}

    bag = [float(np.sort(p['red'])[-max(1, math.ceil(len(p['red']) * cfg['topk_frac'])):].mean()) for p in test_preds]
    m['bag_auc'] = float(roc_auc_score([c['y'] for c in test_clips], bag))

    yt, s_red, s_alarm = [], [], []
    for c, p in zip(test_clips, test_preds):
        if c['y'] == 1 and not c['intervals']:
            continue
        yt.append(frame_truth(c))
        s_red.append(window_scores_to_frames(c, p['red']))
        s_alarm.append(window_scores_to_frames(c, p['alarm']))
    yt = np.concatenate(yt)
    m['frame_ap'] = float(average_precision_score(yt, np.concatenate(s_red)))
    m['frame_auc'] = float(roc_auc_score(yt, np.concatenate(s_red)))
    m['frame_ap_alarm'] = float(average_precision_score(yt, np.concatenate(s_alarm)))

    # Two operating points for the alarm (>= Escalating): the natural one
    # (posterior 0.5) and one calibrated on VALIDATION normal videos.
    val_norm_max = np.array([p['alarm'].max() for c, p in zip(val_clips, val_preds) if c['y'] == 0])
    ops = {'p50': 0.5,
           'cal': max(float(np.quantile(val_norm_max, 1 - cfg['video_fa_target'])), cfg['tau_floor'])}
    test_norm = [p['alarm'] for c, p in zip(test_clips, test_preds) if c['y'] == 0]
    normal_hours = sum(len(s) for s in test_norm) * STRIDE_S / 3600.0
    _near, far_alarm, _cap = pre_onset_windows(lambda c: by_id[c['id']]['alarm'])
    for tag, tau in ops.items():
        m[f'{tag}_tau'] = tau
        fa_videos = [bool((s >= tau).any()) for s in test_norm]
        m[f'{tag}_normal_video_fa'] = float(np.mean(fa_videos))
        m[f'{tag}_normal_window_rate'] = float(np.mean(np.concatenate(test_norm) >= tau))
        runs = sum(int((np.diff(np.r_[0, (s >= tau).astype(int)]) == 1).sum()) for s in test_norm)
        m[f'{tag}_false_alarms_per_hour'] = runs / max(normal_hours, 1e-9)
        # Practical leakage: how often is the alarm on long BEFORE a riot, vs. on normal videos?
        m[f'{tag}_far_window_rate'] = float(np.mean(far_alarm >= tau)) if len(far_alarm) else float('nan')
        rep, costs = lead_time_report(first_onsets, by_id, tau, cfg)
        for k, v in rep.items():
            m[f'{tag}_{k}'] = v
        costs = costs + [cfg['c_fa'] if fa else 0.0 for fa in fa_videos]
        m[f'{tag}_avg_cost'] = float(np.mean(costs))
        rep_all, _ = lead_time_report(all_onsets, by_id, tau, cfg)
        m[f'{tag}_early_all_onsets'] = rep_all['early']
        m[f'{tag}_n_all_onsets'] = rep_all['n_events']

    for k, v in pre_onset_sets(lambda c: by_id[c['id']]['alarm']).items():
        m[f'alarm_{k}'] = v
    for k, v in pre_onset_sets(lambda c: by_id[c['id']]['red']).items():
        m[f'red_{k}'] = v
    if test_preds[0]['esc'] is not None:
        for k, v in pre_onset_sets(lambda c: by_id[c['id']]['esc']).items():
            m[f'esc_{k}'] = v
    return m, by_id

In [ ]:
RUN_ARMS = None   # e.g. ['baseline', 'ordinal_abs', 'full'] to skip full_adv; None = all in CFG
arms = [a for a in (RUN_ARMS or CFG['arms']) if a != 'full_adv' or MEDIUM_OK]
if len(arms) < len(CFG['arms']):
    print("Skipping full_adv: not enough movie and YouTube normal training clips yet.")

# Resumable: every finished (arm, seed) is appended to the CSV immediately,
# and runs already in the CSV are skipped. A disconnect, or switching
# runtime (CPU -> GPU restarts Python), loses at most the run in progress.
results_path = os.path.join(OUT_DIR, 'escalation_results_per_seed.csv')
prev = pd.read_csv(results_path) if os.path.exists(results_path) else pd.DataFrame(columns=['arm', 'seed'])
done = {(a, int(s)) for a, s in zip(prev['arm'], prev['seed'])}
if done:
    print(f"Resuming: {len(done)} runs already saved -> {sorted(done)}")
rows, kept_models, kept_preds = prev.to_dict('records'), {}, {}
for arm in arms:
    for seed in CFG['seeds']:
        ckpt = os.path.join(OUT_DIR, f'escalation_{arm}_seed{seed}.pt')
        if (arm, seed) in done:
            # Re-create the first-seed predictions for the Step 8 timelines
            # from the saved checkpoint (cheap: evaluation only, no training).
            if seed == CFG['seeds'][0] and os.path.exists(ckpt):
                model = EscalationNet(IN_DIM, arm, CFG).to(DEVICE)
                model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
                model.eval()
                kept_models[arm], kept_preds[arm] = model, evaluate(model, CFG)[1]
            continue
        model = train_arm(arm, seed, CFG)
        metrics, by_id = evaluate(model, CFG)
        rows.append(dict(arm=arm, seed=seed, **metrics))
        pd.DataFrame(rows).to_csv(results_path, index=False)
        print(f"  -> {arm} seed {seed}: frame AP {metrics['frame_ap']:.4f} | "
              f"@0.5: early {metrics['p50_early']:.2f} premature {metrics['p50_premature']:.2f} "
              f"late {metrics['p50_late']:.2f} miss {metrics['p50_miss']:.2f} "
              f"normal-video FA {metrics['p50_normal_video_fa']:.2f} | "
              f"alarm AUC near {metrics['alarm_auc_near_all']:.3f} far {metrics['alarm_auc_far_all']:.3f}  [saved]")
        if seed == CFG['seeds'][0]:
            kept_models[arm], kept_preds[arm] = model, by_id
            torch.save(model.state_dict(), ckpt)

results = pd.DataFrame(rows)
arms = [a for a in arms if a in set(results['arm'])]

## Step 7 — Summary across seeds

Mean ± std over seeds. A difference between two arms smaller than
about one std is **not** a finding (same rule as every earlier result
in `docs/findings.md`).

In [ ]:
SHOW = [
    ('frame_ap', 'frame AP (Critical)'), ('frame_auc', 'frame AUC'), ('bag_auc', 'video AUC'),
    ('p50_early', 'early warning @0.5'), ('p50_median_lead_s', 'median lead (s) @0.5'),
    ('p50_premature', 'premature @0.5'), ('p50_late', 'late @0.5'),
    ('p50_median_delay_s', 'median delay when late (s) @0.5'), ('p50_miss', 'miss @0.5'),
    ('p50_normal_video_fa', 'normal videos with any alarm @0.5'),
    ('p50_false_alarms_per_hour', 'false alarms / hour of normal video @0.5'),
    ('p50_normal_window_rate', 'alarm-on rate, normal windows @0.5'),
    ('p50_far_window_rate', 'alarm-on rate, far pre-onset windows @0.5'),
    ('p50_avg_cost', 'avg cost @0.5'),
    ('cal_tau', 'calibrated threshold'), ('cal_early', 'early warning @cal'),
    ('cal_median_lead_s', 'median lead (s) @cal'), ('cal_premature', 'premature @cal'),
    ('cal_late', 'late @cal'), ('cal_miss', 'miss @cal'),
    ('cal_normal_video_fa', 'normal videos with any alarm @cal'), ('cal_avg_cost', 'avg cost @cal'),
    ('alarm_auc_near_all', 'alarm AUC near-onset'), ('alarm_auc_far_all', 'alarm AUC far pre-onset'),
    ('alarm_auc_far_yt', 'alarm AUC far (YouTube controls)'),
    ('esc_auc_near_all', 'escalation AUC near-onset'), ('esc_auc_far_all', 'escalation AUC far pre-onset'),
    ('esc_auc_far_yt', 'escalation AUC far (YouTube controls)'),
    ('red_auc_far_all', 'critical AUC far pre-onset'),
]
table = {}
for arm in arms:
    sub = results[results['arm'] == arm]
    table[arm] = {label: (f"{sub[k].mean():.3f} ± {sub[k].std(ddof=0):.3f}" if k in sub and sub[k].notna().any() else '—')
                  for k, label in SHOW}
summary_table = pd.DataFrame(table)
n_ev = int(results['p50_n_events'].iloc[0])
header = (f"Escalation-stage model — official test split, {len(CFG['seeds'])} seeds, "
          f"{len(train_clips)} train / {len(val_clips)} val / {len(test_clips)} test clips "
          f"({sum(1 - c['y'] for c in test_clips)} test normals). "
          f"Lead-time rows computed on n = {n_ev} first onsets with >= {CFG['min_pre_s']:.0f}s of calm before them.")
print(header + "\n")
print(summary_table.to_string())
print("\nRaw-feature probe for reference (Step 3):",
      {k: round(v, 3) for k, v in raw_probe.items() if k.startswith('auc')})

with open(os.path.join(OUT_DIR, 'escalation_summary.md'), 'w') as fh:
    fh.write(header + "\n\n| metric | " + " | ".join(arms) + " |\n|---|" + "---|" * len(arms) + "\n")
    for _k, label in SHOW:
        fh.write(f"| {label} | " + " | ".join(table[a][label] for a in arms) + " |\n")
    fh.write("\nRaw-feature probe: " + json.dumps({k: round(v, 3) if isinstance(v, float) else v
                                                    for k, v in raw_probe.items()}) + "\n")
    fh.write("\nConfig: " + json.dumps(CFG) + "\n")
print(f"\nSaved -> {OUT_DIR}/escalation_summary.md and escalation_results_per_seed.csv")

## Step 8 — Green / amber / red timelines

Stage posterior over time for test riot clips that have calm footage
before the riot (first seed of the chosen arm). The hatched band is the
annotated riot interval; the dashed line is its onset. This is the
static version of the live indicator Geetanjali asked for.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

VIZ_ARM = 'full' if 'full' in kept_preds else arms[-1]
viz = sorted({c['id']: (c, ev) for c, ev in first_onsets}.values(), key=lambda ce: -ce[1][1])[:8]
if viz and kept_preds[VIZ_ARM][viz[0][0]['id']]['stages'] is not None:
    fig, axes = plt.subplots(len(viz), 1, figsize=(11, 1.5 * len(viz)), squeeze=False)
    for ax, (c, ev) in zip(axes[:, 0], viz):
        st, t = kept_preds[VIZ_ARM][c['id']]['stages'], c['times']
        ax.stackplot(t, st[:, 0], st[:, 1], st[:, 2], colors=['#3a9d5d', '#e8a33d', '#d1453b'], linewidth=0)
        for s, e in c['intervals']:
            ax.axvspan(s / c['fps'], e / c['fps'], facecolor='none', edgecolor='black', hatch='//', alpha=0.35)
        ax.axvline(ev[1], color='black', linestyle='--', linewidth=1)
        ax.set_xlim(t[0], t[-1]); ax.set_ylim(0, 1); ax.set_yticks([])
        ax.set_title(c['id'][:70], fontsize=8, loc='left')
    axes[-1, 0].set_xlabel('seconds')
    fig.tight_layout()
    out_png = os.path.join(OUT_DIR, f'escalation_timelines_{VIZ_ARM}.png')
    fig.savefig(out_png, dpi=130)
    print('Saved ->', out_png)
else:
    print('No eligible clips (or baseline arm only) -- nothing to draw.')

## What to send back

Paste the printed summary table (or `escalation_summary.md`) and the
class/split counts from Step 1. Before any of it goes into
`docs/findings.md`:

- If `far pre-onset` AUCs are well above 0.5 for an arm, that arm's
  "early warnings" are at least partly source/scene leakage.
- `early warning` must be read together with `premature` and with the
  false-alarm column -- a model can buy early warnings by being amber
  all the time.
- Compare arms only beyond the seed std. With ~25 onsets, one clip is
  4 percentage points.